In [4]:
import pandas as pd
import plotly.graph_objects as go

df = pd.read_csv("../DataSets/wellbeing_long_with_domain.csv")

# Lista di paesi europei + USA
allowed_countries = ["Austria", "Belgium","Bulgaria","Croatia", "Denmark","Estonia", "Finland", "France",
                     "Germany", "Greece","Hungary","Iceland", "Ireland", "Italy","Latvia","Lithuania", "Luxembourg", "Netherlands",
                     "Norway","Poland", "Portugal", "Romania","Slovenia","Spain", "Sweden", "United Kingdom", ]

df = df[df["Reference area"].isin(allowed_countries)]

countries = sorted(df["Reference area"].unique())
years = sorted(df["TIME_PERIOD"].unique())

domain_colors = {
    "Income & wealth":        "#1f77b4",
    "Work & job quality":     "#ff7f0e",
    "Housing":                "#2ca02c",
    "Health":                 "#d62728",
    "Knowledge & skills":     "#9467bd",
    "Social connections":     "#8c564b",
    "Safety":                 "#e377c2",
    "Subjective well-being":  "#7f7f7f",
}

def get_year_arrays(country):
    r_list, theta_list, color_list, hover_list = [], [], [], []
    for y in years:
        d = df[(df["Reference area"] == country) & (df["TIME_PERIOD"] == y)].copy()
        d = d.sort_values(["Domain", "Metric"]).reset_index(drop=True)
        theta = d["Metric"].tolist()
        r = d["Value"].tolist()
        colors = d["Domain"].map(domain_colors).tolist()
        theta_list.append(theta)
        r_list.append(r)
        color_list.append(colors)
        hover_list.append(theta)
    return r_list, theta_list, color_list, hover_list

# ---------- init ----------
init_country = countries[0]
r_init, theta_init, color_init, hover_init = get_year_arrays(init_country)

traces = []
for i, y in enumerate(years):
    traces.append(go.Barpolar(
        r=r_init[i],
        theta=theta_init[i],
        marker=dict(color=color_init[i], line=dict(color="white", width=1)),
        hovertext=hover_init[i],
        hovertemplate="%{hovertext}<br>Value=%{r:.3f}<extra></extra>",
        visible=(i == 0)
    ))

fig = go.Figure(data=traces)

# ---------- slider per anno ----------
steps = []
for i, y in enumerate(years):
    steps.append(dict(
        method="update",
        args=[{"visible": [j == i for j in range(len(years))]}],
        label=str(y)
    ))

sliders = [dict(
    active=0,
    currentvalue={"prefix": "Year: ", "font": {"color": "white"}},
    pad={"t": 50, "b": 10},
    steps=steps,
    font={"color": "white"},
    bgcolor="#333333",
    bordercolor="#555555",
    borderwidth=1,
    ticklen=5,
    tickcolor="white"
)]

# ---------- dropdown per nazione ----------
dropdown_options = []
for c in countries:
    r_c, theta_c, color_c, hover_c = get_year_arrays(c)
    
    # Create visibility array
    visible_array = [i == 0 for i in range(len(years))]
    
    dropdown_options.append(dict(
        label=c,
        method="update",
        args=[
            {
                "r": [r_c[i] for trace, i in zip(fig.data, range(len(years)))],
                "theta": [theta_c[i] for trace, i in zip(fig.data, range(len(years)))],
                "marker.color": [color_c[i] for trace, i in zip(fig.data, range(len(years)))],
                "hovertext": [hover_c[i] for trace, i in zip(fig.data, range(len(years)))],
                "visible": visible_array
            },
            {
                "title": dict(
                    text=f"<span style='font-size: 24px; font-weight: bold;'>{c}</span><br>"
                         f"<span style='font-size: 16px;'>Year: {years[0]}</span>",
                    font=dict(color="white", size=24),
                    x=0.5,
                    xanchor="center"
                ),
                "sliders": sliders
            }
        ],
    ))

# Custom layout con tema scuro migliorato
fig.update_layout(
    title=dict(
        text=f"<span style='font-size: 24px; font-weight: bold;'>{init_country}</span><br>"
             f"<span style='font-size: 16px;'>Year: {years[0]}</span>",
        font=dict(color="white", size=24),
        x=0.5,
        xanchor="center",
        y=0.95
    ),
    template="plotly_dark",
    plot_bgcolor="#1a1a1a",
    paper_bgcolor="#1a1a1a",
    font=dict(color="white"),
    
    polar=dict(
        bgcolor="#2d2d2d",
        angularaxis=dict(
            type="category", 
            direction="clockwise", 
            rotation=90,
            gridcolor="#555555",
            linecolor="#777777",
            tickfont=dict(color="white", size=12)
        ),
        radialaxis=dict(
            range=[0, 1], 
            ticks="outside",
            gridcolor="#555555",
            linecolor="#777777",
            tickfont=dict(color="white")
        )
    ),
    
    # Slider position
    sliders=sliders,
    
    # Dropdown migliorato
    updatemenus=[dict(
        type="dropdown",
        x=0.03,
        y=1.05,
        xanchor="left",
        yanchor="top",
        showactive=True,
        bgcolor="rgba(40, 40, 40, 0.9)",
        bordercolor="#555555",
        borderwidth=1,
        font=dict(color="white", size=12),
        buttons=dropdown_options,
        pad={"r": 10, "t": 10, "b": 10},
        direction="down",
        active=0
    )],
    
    # Aggiungi titolo dropdown
    annotations=[
        dict(
            text="Select Country:",
            x=0.03,
            y=1.12,
            xref="paper",
            yref="paper",
            showarrow=False,
            font=dict(color="white", size=14),
            xanchor="left"
        )
    ],
    
    # Margini per evitare sovrapposizioni
    margin=dict(t=150, b=80, l=50, r=50),
)

fig.show()
fig.write_html("../vizualizations/radial_chart_dark_dropdown_left.html")